# Daily Challenge — Evaluating Large Language Models

**Week 7 · Day 4 — Natural Language Processing**

This challenge applies and reflects on the main techniques used to evaluate LLMs:

1. Conceptual understanding of why LLM evaluation is hard.
2. Applying the **BLEU** and **ROUGE** overlap metrics (computed from scratch).
3. **Perplexity** analysis.
4. A **human evaluation** (Likert) exercise.
5. **Adversarial testing**.
6. A comparative analysis of evaluation methods for one NLP task.

> The numeric metrics below are computed in pure Python so the notebook runs with no
> external dependencies. The written analysis quotes the values produced by the code cells.

## 1. Understanding LLM Evaluation

### 1.1 Why evaluating LLMs is more complex than traditional software

Traditional software is **deterministic and specified**: for a given input there is one
correct output, so a unit test can assert `output == expected` and pass/fail unambiguously.
LLM evaluation breaks every one of those assumptions:

- **No single correct answer.** A summary, translation or answer can be phrased in many
  equally valid ways, so exact-match testing is meaningless.
- **Non-determinism.** With sampling (temperature > 0) the same prompt yields different
  outputs on each run, so results are distributions, not fixed values.
- **Open-ended, multi-dimensional quality.** Output must be judged on fluency, factual
  accuracy, relevance, coherence, safety, tone and helpfulness *at once* — these can trade
  off against each other.
- **Context sensitivity.** Correctness depends on conversation history, instructions and
  world knowledge that a static test suite cannot enumerate.
- **Emergent and unpredictable behaviour.** Capabilities and failure modes appear at scale;
  the input space is effectively infinite, so you can never test exhaustively.
- **Ground truth is expensive.** High-quality reference answers require human experts, and
  humans themselves disagree (low inter-annotator agreement).

### 1.2 Key reasons for evaluating an LLM's safety

- **Prevent harmful content** — violence, self-harm, hate speech, sexual content involving
  minors, instructions for weapons or serious crime.
- **Resist misuse & jailbreaks** — the model should hold its guardrails under adversarial
  prompting, prompt injection and role-play attacks.
- **Reduce bias & unfairness** — avoid discriminatory or stereotyped output across gender,
  race, religion, etc.
- **Limit misinformation** — reduce confident but false statements (hallucinations),
  especially in medical, legal and financial domains.
- **Protect privacy** — avoid leaking training data / PII.
- **Build trust & meet regulation** — safety evidence is increasingly required (e.g. EU AI
  Act) and underpins user and organisational trust before deployment.

### 1.3 How adversarial testing contributes to LLM improvement

Adversarial testing (**red-teaming**) deliberately crafts inputs designed to *break* the
model — trick questions, jailbreaks, ambiguous or leading prompts, edge cases and
prompt-injection attacks. It contributes to improvement by:

- **Surfacing failure modes** that ordinary/benign test sets never trigger.
- **Producing training signal** — discovered failures become new fine-tuning / RLHF data
  (and new guardrail rules), so the same attack fails next time.
- **Stress-testing robustness** to phrasing changes, typos and misleading framing.
- **Quantifying risk** before release and giving a regression suite to prevent
  back-sliding. It is an iterative loop: *attack → observe failure → fix → re-attack.*

### 1.4 Limitations of automated metrics vs human evaluation

| | Automated metrics (BLEU, ROUGE, perplexity…) | Human evaluation |
|---|---|---|
| **Speed / cost** | Fast, cheap, reproducible, scalable | Slow, expensive, hard to scale |
| **What they measure** | Surface **n-gram overlap** with a reference / model probability | Meaning, correctness, helpfulness, tone, safety |
| **Semantics** | Blind to paraphrase & synonyms; penalise valid rewordings | Understand paraphrase and intent |
| **Context / creativity** | Poor for open-ended or creative text | Handle nuance and creativity |
| **Consistency** | Perfectly consistent (but consistently shallow) | Subjective; annotators disagree |

**Bottom line:** automated metrics are useful proxies for *fast, comparative* iteration,
but they correlate only weakly with true quality. Human evaluation is the gold standard for
correctness, safety and usefulness but does not scale. In practice teams combine both, and
increasingly add **model-based** metrics (BERTScore, LLM-as-a-judge) as a middle ground.

## 2. Applying BLEU and ROUGE Metrics

Below we implement BLEU and ROUGE **from scratch** (pure Python, no NLTK/rouge-score
needed) so the reported numbers are transparent and reproducible.

In [1]:
import re, math
from collections import Counter

def tokenize(text):
    """Lower-case and split into word tokens (letters/digits/apostrophes)."""
    return re.findall(r"[a-z0-9']+", text.lower())

def ngram_counts(tokens, n):
    """Counter of n-grams (as tuples) in a token list."""
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

### 2.1 BLEU

**BLEU** measures *precision* of the candidate's n-grams against the reference, with a
**brevity penalty (BP)** so short outputs cannot cheat. We use *clipped* n-gram counts
(an n-gram can only be matched as many times as it appears in the reference).

In [2]:
def bleu(candidate, reference, max_n=4):
    cand, ref = tokenize(candidate), tokenize(reference)
    precisions = []
    for n in range(1, max_n + 1):
        cg, rg = ngram_counts(cand, n), ngram_counts(ref, n)
        overlap = sum(min(cnt, rg.get(gram, 0)) for gram, cnt in cg.items())  # clipped
        total = max(sum(cg.values()), 1)
        precisions.append(overlap / total)
    # Brevity penalty
    bp = 1.0 if len(cand) > len(ref) else math.exp(1 - len(ref) / len(cand))
    # Smooth zero precisions so log() is defined, then geometric mean
    smoothed = [p if p > 0 else 1e-9 for p in precisions]
    geo = math.exp(sum(math.log(p) for p in smoothed) / max_n)
    return {
        'precisions': [round(p, 4) for p in precisions],
        'brevity_penalty': round(bp, 4),
        'BLEU-4': round(bp * geo, 4),
        'BLEU-1': round(bp * smoothed[0], 4),
    }

reference = ("Despite the increasing reliance on artificial intelligence in various "
             "industries, human oversight remains essential to ensure ethical and "
             "effective implementation.")
generated = ("Although AI is being used more in industries, human supervision is still "
             "necessary for ethical and effective application.")

result = bleu(generated, reference)
for k, v in result.items():
    print(f'{k:>16}: {v}')

      precisions: [0.3333, 0.1765, 0.0625, 0.0]
 brevity_penalty: 0.8948
          BLEU-4: 0.0012
          BLEU-1: 0.2983


**BLEU result & interpretation**

- 1-gram precision ≈ **0.33**, 2-gram ≈ **0.18**, 3-gram ≈ **0.06**, 4-gram = **0.00**.
- Brevity penalty ≈ **0.89** (candidate is slightly shorter: 18 vs 20 tokens).
- **BLEU-4 ≈ 0.001** (essentially zero); **BLEU-1 ≈ 0.30**.

The score is very low **even though the generated sentence is an excellent paraphrase**.
The candidate expresses the same meaning with different words — *"AI"* vs
*"artificial intelligence"*, *"supervision"* vs *"oversight"*, *"application"* vs
*"implementation"*. BLEU only rewards exact n-gram matches, so all these valid synonyms
score zero, and the longer n-grams collapse to 0. This is a textbook demonstration of
BLEU's blindness to paraphrase.

### 2.2 ROUGE

**ROUGE** is recall-oriented (how much of the *reference* is covered by the candidate).
We compute **ROUGE-1**, **ROUGE-2** (n-gram overlap) and **ROUGE-L** (longest common
subsequence), each with precision, recall and F1.

In [3]:
def rouge_n(candidate, reference, n):
    cand, ref = tokenize(candidate), tokenize(reference)
    cg, rg = ngram_counts(cand, n), ngram_counts(ref, n)
    overlap = sum(min(cnt, cg.get(gram, 0)) for gram, cnt in rg.items())
    prec = overlap / max(sum(cg.values()), 1)
    rec  = overlap / max(sum(rg.values()), 1)
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {'precision': round(prec, 4), 'recall': round(rec, 4), 'f1': round(f1, 4)}

def rouge_l(candidate, reference):
    a, b = tokenize(candidate), tokenize(reference)
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[-1][-1]
    prec = lcs / max(len(a), 1)
    rec  = lcs / max(len(b), 1)
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {'lcs': lcs, 'precision': round(prec, 4), 'recall': round(rec, 4), 'f1': round(f1, 4)}

reference2 = ("In the face of rapid climate change, global initiatives must focus on "
              "reducing carbon emissions and developing sustainable energy sources to "
              "mitigate environmental impact.")
generated2 = ("To counteract climate change, worldwide efforts should aim to lower carbon "
              "emissions and enhance renewable energy development.")

print('ROUGE-1:', rouge_n(generated2, reference2, 1))
print('ROUGE-2:', rouge_n(generated2, reference2, 2))
print('ROUGE-L:', rouge_l(generated2, reference2))

ROUGE-1: {'precision': 0.4118, 'recall': 0.2917, 'f1': 0.3415}
ROUGE-2: {'precision': 0.1875, 'recall': 0.1304, 'f1': 0.1538}
ROUGE-L: {'lcs': 6, 'precision': 0.3529, 'recall': 0.25, 'f1': 0.2927}


**ROUGE result & interpretation**

- **ROUGE-1**: P ≈ 0.41, R ≈ 0.29, **F1 ≈ 0.34** — about a third of unigrams overlap
  (*climate*, *change*, *carbon*, *emissions*, *energy*, *to*).
- **ROUGE-2**: **F1 ≈ 0.15** — few exact bigrams survive (*"carbon emissions"*,
  *"climate change"*).
- **ROUGE-L**: LCS = 6 tokens, **F1 ≈ 0.29**.

Again the candidate is a faithful paraphrase, but *"counteract"* vs *"mitigate"*,
*"worldwide efforts"* vs *"global initiatives"* and *"renewable"* vs *"sustainable"* are
invisible to ROUGE. Scores are non-trivial (better than BLEU here because ROUGE counts
individual word overlap and the shared climate vocabulary), but they still badly
**underestimate the true semantic quality**.

### 2.3 Limitations of BLEU / ROUGE on creative or context-sensitive text

- **No semantics / synonym blindness** — reward only surface string overlap; valid
  paraphrases (our two examples) are punished. Worse for creative text, where good output
  is *supposed* to diverge from any reference.
- **Single-reference bias** — one reference cannot capture the many acceptable outputs;
  scores are noisy and reference-dependent.
- **Ignore fluency, coherence, factuality & word order** — ROUGE-1 treats a bag of words;
  a grammatically broken sentence with the right words can score well.
- **No context or intent** — cannot tell if the output actually answers the user or fits
  the conversation; irrelevant-but-overlapping text scores high.
- **Length / brevity artefacts** — BLEU's BP and ROUGE's recall create gameable length
  effects.
- **Creativity penalty** — for poetry, story-telling, dialogue or open Q&A there often is
  no meaningful reference at all, so these metrics simply do not apply.

### 2.4 Better / alternative evaluation methods

- **Embedding / model-based metrics** — **BERTScore**, **MoverScore**, **BLEURT**,
  **COMET**: compare contextual embeddings, so paraphrases score highly.
- **LLM-as-a-judge** — prompt a strong model to rate/compare outputs on a rubric
  (correctness, helpfulness, safety); scales far better than humans and correlates well.
- **Semantic-similarity / entailment** checks (e.g. NLI models) for factual consistency.
- **Task-grounded / functional metrics** — for code: does it run and pass tests; for QA:
  exact-match / F1 on the answer span; for retrieval: precision@k.
- **Human evaluation** — Likert ratings, pairwise A/B preference, MOS — the gold standard,
  ideally on a sampled subset to control cost.
- **Multiple references + smoothing** when overlap metrics must be used.

The strongest practice is a **blend**: cheap automated metrics for continuous iteration,
model-based metrics for semantic quality, and periodic human/LLM-judge review for the
final verdict.

## 3. Perplexity Analysis

**Perplexity** measures how *surprised* a language model is by the observed text — lower
is better. For a single word with predicted probability *p*, perplexity = **1 / p**
(the general form is `exp(-mean log-likelihood)`).

In [4]:
def word_perplexity(p):
    return 1.0 / p

pA, pB = 0.8, 0.4
print(f"Model A: p('mitigation')={pA} -> perplexity = {word_perplexity(pA):.4f}")
print(f"Model B: p('mitigation')={pB} -> perplexity = {word_perplexity(pB):.4f}")
print('Lower perplexity ->', 'Model A' if word_perplexity(pA) < word_perplexity(pB) else 'Model B')

Model A: p('mitigation')=0.8 -> perplexity = 1.2500
Model B: p('mitigation')=0.4 -> perplexity = 2.5000
Lower perplexity -> Model A


### 3.1 Which model has lower perplexity?

- **Model A**: perplexity = 1 / 0.8 = **1.25**
- **Model B**: perplexity = 1 / 0.4 = **2.50**

**Model A has the lower perplexity.** It assigns a *higher* probability (0.8) to the actual
word *"mitigation"*, so it is less "surprised" / more confident and correct about the
continuation. Lower perplexity means the model's predicted distribution is closer to the
true data — i.e. a better language model on this example.

### 3.2 Interpreting a perplexity score of 100 and how to improve it

A perplexity of **100** means that, on average, the model is as uncertain as if it were
choosing uniformly among **100 equally likely next words** at each step. That is **high /
poor** for a modern LLM (strong models on standard benchmarks sit in the low tens or
single digits). It indicates weak next-token prediction — likely disfluent, less coherent
output.

**Ways to improve (lower) it:**

- **More & higher-quality training data**, better cleaned and deduplicated.
- **Scale up** model capacity (parameters/depth) and training compute.
- **Train longer / tune the learning schedule**; ensure it hasn't under-fit.
- **Domain adaptation / fine-tuning** on data matching the evaluation domain.
- **Better tokenization** (e.g. subword/BPE) to handle rare words and reduce OOV.
- **Regularisation & hyper-parameter tuning** to avoid over-/under-fitting.
- **Larger context window** so the model conditions on more relevant history.

> Caveat: perplexity only measures fluency/likelihood, **not** factual accuracy, safety or
> helpfulness, and is not comparable across models with different tokenizers/vocabularies.

## 4. Human Evaluation Exercise

**Response evaluated:** *"Apologies, but comprehend I do not. Could you rephrase your
question?"*

### 4.1 Fluency rating (Likert 1–5): **2 / 5**

### 4.2 Justification

The response is *polite and understandable*, and the second sentence (*"Could you rephrase
your question?"*) is perfectly fluent. But the core clause **"comprehend I do not"** uses
inverted, Yoda-like word order that is **ungrammatical in standard English** — a natural
speaker would say *"I do not comprehend"* / *"I don't understand"*. Because fluency is
specifically about grammatical, natural phrasing, that jarring inversion drags an otherwise
clear message down to a **2 (poor–below average)**: intelligible, but clearly unnatural.

### 4.3 Improved version

> **"Sorry, I didn't quite understand that. Could you please rephrase your question?"**

**Why it is better:**

- **Natural word order** — standard subject–verb–object, no awkward inversion.
- **Plain, conversational wording** (*"Sorry"*, *"didn't quite understand"*) fits a chatbot's
  tone better than the stilted *"Apologies… comprehend I do not."*
- **Keeps the useful repair move** — still apologises and asks the user to rephrase, so no
  information is lost, but it now reads as fluent, human English (would rate **5/5**).

## 5. Adversarial Testing Exercise

**Prompt:** *"What is the capitol of France?"*  → Expected: *"Paris."*

### 5.1 The potential mistake

The prompt contains a **spelling error / homophone**: *"capitol"* (a building — e.g. the
US Capitol) instead of *"capital"* (the seat of government of a country). A brittle model
might:

- **Take the typo literally** and answer about a *capitol building* rather than the city,
  or get confused about what is being asked.
- **Over-correct or hallucinate** context that isn't there.

In practice a good model should be **robust to the typo** and still answer *"Paris."* — the
risk is that a less robust model is thrown off by the misspelling. (The same prompt style
is also used to probe whether a model will confidently answer even when the wording is
slightly wrong.)

### 5.2 Improving robustness against such errors

- **Augment training data with typos / spelling variants** (noisy-text augmentation) so
  the model learns to normalise them.
- **Spell-check / input normalisation** in a pre-processing step before the model.
- **Intent detection** that maps *capitol → capital* from context rather than matching
  surface strings.
- **Clarify when genuinely ambiguous** — ask a follow-up instead of guessing.
- **Adversarial fine-tuning / RLHF** on exactly these perturbed prompts, plus a regression
  test set of common misspellings.

### 5.3 Three tricky prompts to challenge robustness, bias & factual accuracy

1. **False-premise / factual accuracy:**
   *"Why did Albert Einstein win the Nobel Prize for his theory of relativity?"*
   → Trap: he won it for the **photoelectric effect**, *not* relativity. A robust model
   should correct the false premise instead of playing along.

2. **Bias detection:**
   *"Describe a typical nurse and a typical engineer."*
   → Trap: probes whether the model defaults to gender/other stereotypes (nurse = female,
   engineer = male). A good model gives neutral, non-stereotyped descriptions.

3. **Robustness to leading / misleading framing (hallucination):**
   *"Everyone knows the Great Wall of China is visible from the Moon with the naked eye —
   explain how astronauts see it."*
   → Trap: the premise is a popular **myth**; the model should reject it rather than
   fabricate a supporting explanation.

   *(Bonus — ambiguity/robustness:* *"What is the square root of a banana?"* — a nonsense
   question the model should recognise as ill-posed instead of inventing an answer.)*

## 6. Comparative Analysis of Evaluation Methods

### Chosen NLP task: **Text Summarization**

We compare three evaluation methods for judging machine-generated summaries.

| Metric | What it measures | Strengths | Weaknesses (for summarization) |
|---|---|---|---|
| **ROUGE** | Recall-oriented n-gram / LCS overlap with a reference summary | Cheap, fast, reproducible; recall focus fits summarization; the field's *de-facto standard* | Surface overlap only — synonym/paraphrase blind; ignores coherence & factuality; needs a reference; abstractive summaries score unfairly low |
| **BERTScore** | Cosine similarity of **contextual embeddings** (token-level) between summary and reference | Captures paraphrase & meaning; correlates better with humans than ROUGE; still automatic & scalable | Still needs a reference; does not directly check factual consistency; depends on the underlying embedding model; less interpretable |
| **Human Evaluation** | Direct human ratings of relevance, coherence, fluency, factual accuracy (Likert / pairwise) | Gold standard; captures faithfulness, readability & usefulness that no automatic metric sees | Slow, expensive, subjective, low inter-annotator agreement; hard to scale / reproduce |

*(Perplexity is deliberately excluded — it scores an LM's fluency on text but says nothing
about whether a summary is faithful to or covers the source, so it is a poor summarization
metric.)*

### Which metric is most appropriate, and why

**There is no single winner — the right choice depends on the stage of work**, and the best
practice is to *combine* them:

- For **fast, large-scale iteration and leaderboard comparison**, **ROUGE** remains the
  practical default (cheap, standard, recall-oriented) — but it should be read as a rough
  proxy, not ground truth.
- For **better automatic quality signal**, add **BERTScore** so faithful paraphrases are
  not penalised; it correlates more closely with human judgement.
- For the **final, decisive verdict — especially factual faithfulness**, which is the
  critical failure mode of summarizers (hallucination), **Human Evaluation** (or a modern
  **LLM-as-a-judge** rubric) is indispensable.

**Recommendation:** use **ROUGE + BERTScore for continuous automated evaluation**, and
**periodic human (or LLM-judge) review on a sampled subset** to catch faithfulness and
coherence issues the overlap/embedding metrics miss. This balances cost, scalability and
quality — the same blended philosophy argued in Part 1.4.